In [17]:
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor

import matplotlib.pyplot as plt

In [18]:
df = pd.read_csv(
    "Reduced_Metallophilic_Dataset.csv"
)

print(df.shape)

df.head()

df["HX_nearest"] = (
    df["HX_nearest"]
    .fillna(0)
)
hammett_map = {

    "NH2": -0.66,
    "OH" : -0.37,
    "CH3": -0.17,

    "H"  : 0.00,

    "F"  : 0.06,

    "CF3": 0.54,
    "NO2": 0.78
}

df["Hammett_sigma"] = (
    df["R"]
    .map(hammett_map)
)

(144, 37)


In [19]:
print(df.columns.tolist())

['dimer', 'Metal', 'Ligand', 'X', 'R', 'Y_Smiles', 'E', 'd', 'rho', 'laplacian', 'V', 'G', 'V_over_G', 'Unnamed: 13', 'Metal_atomic_number', 'Metal_atomic_radius', 'Metal_electronegativity', 'Metal_polarizability', 'Metal_ionization_energy', 'X_atomic_number', 'X_atomic_radius', 'X_electronegativity', 'X_polarizability', 'X_ionization_energy', 'Shortest_M_N', 'Monomer_Slippage', 'HX_nearest', 'Mol', 'nAromAtom', 'nAromBond', 'nHetero', 'nN', 'nO', 'nF', 'nAtom', 'nHeavyAtom', 'nBase', 'Hammett_sigma']


In [20]:
# =========================================================
# TARGET
# =========================================================

target_column = 'E'


# =========================================================
# QTAIM DESCRIPTORS
# =========================================================

qtaim_columns = [

    'rho',
    'laplacian',
    'V',
    'G',
    'V_over_G'

]


# =========================================================
# GEOMETRY DESCRIPTORS
# =========================================================

geometry_columns = [

    'd',
    'Shortest_M_N',
    'Monomer_Slippage',
    'HX_nearest'

]


# =========================================================
# QM DESCRIPTORS
# =========================================================

qm_columns = (

    qtaim_columns +

    geometry_columns

)


# =========================================================
# ML DESCRIPTORS
# =========================================================

ml_columns = [

    'Metal_atomic_number',
    'Metal_atomic_radius',
    'Metal_electronegativity',
    'Metal_polarizability',
    'Metal_ionization_energy',

    'X_atomic_number',
    'X_atomic_radius',
    'X_electronegativity',
    'X_polarizability',
    'X_ionization_energy',
    
    'nAromAtom',
    'nN',
    'nO',
    'nF',
    'nAtom',
    'Hammett_sigma'

]

# =========================================================
# TARGET VARIABLE
# =========================================================

target_column = "E"

y = df[target_column]

print(y.shape)
# =========================================================
# FULL DESCRIPTOR SET
# =========================================================

full_columns = ( qm_columns + ml_columns )

(144,)


In [21]:
print("QTAIM Features :", len(qtaim_columns))
print("Geometry Features :", len(geometry_columns))
print("QM Features :", len(qm_columns))
print("ML Features :", len(ml_columns))
print("Full Features :", len(full_columns))

QTAIM Features : 5
Geometry Features : 4
QM Features : 9
ML Features : 16
Full Features : 25


In [22]:
all_features = (
    qtaim_columns +
    geometry_columns +
    ml_columns
)

print("Total unique descriptors:",
      len(set(all_features)))

print("Duplicate descriptors:",
      len(all_features) - len(set(all_features)))

Total unique descriptors: 25
Duplicate descriptors: 0


In [23]:
# =========================================================
# FEATURE PREPARATION
# =========================================================

def prepare_features(df, columns):

    X = df[columns].copy()

    scaler = StandardScaler()

    X_scaled = pd.DataFrame(
        scaler.fit_transform(X),
        columns=columns,
        index=df.index
    )

    return X_scaled

In [24]:
cv = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [25]:
models = {

    "Linear Regression":

        LinearRegression(),

    "Random Forest":

        RandomForestRegressor(
            n_estimators=500,
            random_state=42
        ),

    "XGBoost":

        XGBRegressor(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=4,
            random_state=42
        )
}

In [26]:
def evaluate_model(
    model,
    X,
    y,
    cv
):

    fold_r2_scores = []

    all_predictions = []

    fold_number = 1

    for train_idx, test_idx in cv.split(X):

        X_train = X.iloc[train_idx]
        X_test  = X.iloc[test_idx]

        y_train = y.iloc[train_idx]
        y_test  = y.iloc[test_idx]

        model.fit(
            X_train,
            y_train
        )

        y_pred = model.predict(
            X_test
        )

        fold_r2 = r2_score(
            y_test,
            y_pred
        )

        fold_r2_scores.append(
            fold_r2
        )

        fold_results = pd.DataFrame({

            "Fold":
                fold_number,

            "Dimer":
                df.iloc[test_idx]["dimer"].values,

            "Actual_E":
                y_test.values,

            "Predicted_E":
                y_pred,

            "Error":
                y_test.values - y_pred

        })

        all_predictions.append(
            fold_results
        )

        fold_number += 1

    results_df = pd.concat(
        all_predictions,
        ignore_index=True
    )

    results_df["Abs_Error"] = (
        results_df["Error"]
        .abs()
    )

    return {

        "Fold_R2":
            fold_r2_scores,

        "Mean_R2":
            np.mean(
                fold_r2_scores
            ),

        "Std_R2":
            np.std(
                fold_r2_scores
            ),

        "Results":
            results_df

    }

In [27]:
def benchmark_descriptor_set(
    descriptor_name,
    columns,
    models,
    df,
    y,
    cv
):

    X = prepare_features(
        df,
        columns
    )

    summary_rows = []

    detailed_results = {}

    for model_name, model in models.items():

        results = evaluate_model(
            model,
            X,
            y,
            cv
        )

        summary_rows.append({

            "Descriptor_Set":
                descriptor_name,

            "Model":
                model_name,

            "No_of_Features":
                len(columns),

            "Mean_R2":
                results["Mean_R2"],

            "Std_R2":
                results["Std_R2"]

        })

        detailed_results[
            f"{descriptor_name}_{model_name}"
        ] = results["Results"]

    summary_df = pd.DataFrame(
        summary_rows
    )

    return (
        summary_df,
        detailed_results
    )

In [28]:
descriptor_sets = {

    "QTAIM":
        qtaim_columns,

    "QM":
        qm_columns,

    "ML":
        ml_columns,

    "FULL":
        full_columns
}

In [29]:
all_summary_tables = []

all_detailed_results = {}

for descriptor_name, columns in descriptor_sets.items():

    summary_df, detailed_results = (
        benchmark_descriptor_set(
            descriptor_name,
            columns,
            models,
            df,
            y,
            cv
        )
    )

    all_summary_tables.append(
        summary_df
    )

    all_detailed_results.update(
        detailed_results
    )

benchmark_results = pd.concat(
    all_summary_tables,
    ignore_index=True
)

In [30]:
benchmark_results

,Descriptor_Set,Model,No_of_Features,Mean_R2,Std_R2
0,QTAIM,Linear Regression,5,0.569415,0.156068
1,QTAIM,Random Forest,5,0.822970,0.147514
2,QTAIM,XGBoost,5,0.846347,0.111709
3,QM,Linear Regression,9,0.871475,0.031589
4,QM,Random Forest,9,0.950578,0.023133
5,QM,XGBoost,9,0.951419,0.026612
6,ML,Linear Regression,16,0.902455,0.018960
7,ML,Random Forest,16,0.990002,0.005600
8,ML,XGBoost,16,0.996253,0.001506
9,FULL,Linear Regression,25,0.971019,0.008985


In [31]:
benchmark_results.sort_values(
    "Mean_R2",
    ascending=False
)

,Descriptor_Set,Model,No_of_Features,Mean_R2,Std_R2
8,ML,XGBoost,16,0.996253,0.001506
7,ML,Random Forest,16,0.990002,0.005600
10,FULL,Random Forest,25,0.974952,0.014450
11,FULL,XGBoost,25,0.972389,0.013497
9,FULL,Linear Regression,25,0.971019,0.008985
5,QM,XGBoost,9,0.951419,0.026612
4,QM,Random Forest,9,0.950578,0.023133
6,ML,Linear Regression,16,0.902455,0.018960
3,QM,Linear Regression,9,0.871475,0.031589
2,QTAIM,XGBoost,5,0.846347,0.111709


In [32]:
benchmark_results.to_csv(
    "Descriptor_Model_Comparison.csv",
    index=False
)